# LITETime for UCR datasets 65–128

Builds the LITETime ensemble for the second half of the UCR archive from the five single-run passes (no training): for each dataset, the saved model of every pass predicts the test set, the class probabilities are averaged, and the accuracy of the average is LITETime, as in `main.py`.

**Input:** the `lite-second-half-models` dataset (`models/pass<k>/<dataset>/best_model.keras` and `pass_csv/pass_accuracies.csv`).
**Settings:** Internet on, GPU T4 (CPU also works).
**Output:** `litetime_second_half.csv`.

In [2]:
# Setup: same packages as the training runs, and the LITE code for data loading
!pip install -q aeon codecarbon setuptools
!pip install -q -U "keras>=3.6.0"
import os, glob, subprocess, sys
from collections import defaultdict
import numpy as np, pandas as pd, tensorflow as tf, keras, aeon
print("tf", tf.__version__, "| keras", keras.__version__, "| aeon", aeon.__version__)

if not os.path.exists("/kaggle/working/LITE"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MSD-IRIMAS/LITE.git", "/kaggle/working/LITE"], check=True)
sys.path.insert(0, "/kaggle/working/LITE")
from src.utils import load_data

models = defaultdict(dict)
for p in glob.glob("/kaggle/input/**/models/pass*/*/best_model.keras", recursive=True):
    k = int(p.split("/models/pass")[1].split("/")[0])
    models[p.split("/")[-2]][k] = p
acc_csv = pd.read_csv(glob.glob("/kaggle/input/**/pass_accuracies.csv", recursive=True)[0])
csv_acc = {(r.dataset, r["pass"]): r.acc for _, r in acc_csv.iterrows()}
print(sum(len(v) for v in models.values()), "models for", len(models), "datasets")
print("Datasets with fewer than 5 models:", {d: sorted(v) for d, v in models.items() if len(v) < 5})

tf 2.21.0 | keras 3.15.1 | aeon 1.6.0
298 models for 60 datasets
Datasets with fewer than 5 models: {'Lightning2': [1, 2, 3, 4], 'ItalyPowerDemand': [2, 3, 4, 5]}


In [3]:
# LITETime from the saved models; each model is checked against its pass's CSV accuracy
rows = []
for ds in sorted(models):
    xtr, ytr, xte, yte = load_data(ds)
    probs, accs, bad = [], [], []
    for k, p in sorted(models[ds].items()):
        pr = tf.keras.models.load_model(p, compile=False).predict(xte, verbose=0)
        acc = float((pr.argmax(1) == yte).mean())
        if abs(acc - csv_acc.get((ds, k), acc)) > 1e-6:
            bad.append(f"pass{k}: {acc:.4f} vs CSV {csv_acc[(ds, k)]:.4f}")
        probs.append(pr); accs.append(acc)
        tf.keras.backend.clear_session()
    ens = float((np.mean(probs, axis=0).argmax(1) == yte).mean())
    rows.append({"dataset": ds, "models": len(probs), "LITE-mean": np.mean(accs), "LITE-std": np.std(accs),
                 "LITETime": ens, "matches_csv": not bad})
    print(f"{ds:30} models {len(probs)}  LITE-mean {np.mean(accs):.4f}  LITETime {ens:.4f}  {'ok' if not bad else 'MISMATCH ' + '; '.join(bad)}", flush=True)

out = pd.DataFrame(rows)
out.to_csv("/kaggle/working/litetime_second_half.csv", index=False)
print("\nAll models match the pass CSVs:", out.matches_csv.all())

ItalyPowerDemand               models 4  LITE-mean 0.9687  LITETime 0.9660  ok
LargeKitchenAppliances         models 5  LITE-mean 0.8917  LITETime 0.8933  ok
Lightning2                     models 4  LITE-mean 0.7787  LITETime 0.7705  ok
Lightning7                     models 5  LITE-mean 0.8301  LITETime 0.8493  ok
Mallat                         models 5  LITE-mean 0.9390  LITETime 0.9557  ok
Meat                           models 5  LITE-mean 0.9167  LITETime 0.9167  ok
MedicalImages                  models 5  LITE-mean 0.7453  LITETime 0.7684  ok
MiddlePhalanxOutlineAgeGroup   models 5  LITE-mean 0.5377  LITETime 0.5519  ok
MiddlePhalanxOutlineCorrect    models 5  LITE-mean 0.8151  LITETime 0.8419  ok
MiddlePhalanxTW                models 5  LITE-mean 0.5078  LITETime 0.5130  ok
MixedShapesRegularTrain        models 5  LITE-mean 0.9751  LITETime 0.9794  ok
MixedShapesSmallTrain          models 5  LITE-mean 0.9415  LITETime 0.9472  ok
MoteStrain                     models 5  LITE-mean 0